In [1]:
import asyncio
import os

import httpx
from dotenv import load_dotenv

load_dotenv()


URLS = [
    "https://api.github.com/search/repositories?q=terminal+agent",
    "https://api.github.com/search/repositories?q=minecraft",
    "https://api.github.com/search/repositories?q=browser",
]

github_token = os.getenv("GITHUB_TOKEN")
if not github_token:
    raise ValueError("GITHUB_TOKEN environment variable is not set")
HEADERS = {"Authorization": f"Bearer {github_token}"}

In [2]:
async def fetch(client: httpx.AsyncClient, url: str) -> dict:

    response = await client.get(url)

    response.raise_for_status()

    return response.json()


async def main():

    async with httpx.AsyncClient(headers=HEADERS, timeout=20) as client:
        tasks = [fetch(client, url) for url in URLS]

        results = await asyncio.gather(*tasks)

        return results

In [3]:
results = await main()

repositories = []

for result in results:
    items = result.get("items", [])
    items = items[:5]  # Limit to top 5 repositories per query
    for item in items:
        repositories.append(
            {
                "name": item.get("name"),
                "url": item.get("html_url"),
                "description": item.get("description"),
                "language": item.get("language"),
                "stars": item.get("stargazers_count"),
                "watchers": item.get("watchers_count"),
                "forks": item.get("forks_count"),
                "topics": item.get("topics", []),
            }
        )

repositories.sort(key=lambda x: x["stars"], reverse=True)
len(repositories)

15